In [7]:
import pypsa
import os
import pandas as pd

# CI Costs and Revenues
This notebook includes script to evaluate costs and revenues for all the CI components:
- Supply (generators and links)
- Storage units
- CI participating load

N.B.: analysis not suitable for baseline scenarios, since they do not include CI procurement

In [8]:
# 0 - Functions
def get_network_path(scenario, path = False):
    """
    Retrieve the network file path for a given scenario.
    
    - path = False means the default model repo path is used --> go back one level and go to "results/{scenario}/networks/network.nc"
    - path = "custom-path" means the networks are saved in a custom path where the network name is the same as the scenario name --> "custom-path/{scenario}.nc"
    """

    if not path: # Default model repo path: go back one level and go to results/...
        # Define the scenario directory
        scenario_dir = os.path.join("..", f"results/{scenario}/networks")
        if not os.path.exists(scenario_dir):
            print(f"Scenario directory {scenario_dir} not found, skipping.")
            return None
        
        # Get scenario network file
        nc_files = [f for f in os.listdir(scenario_dir) if f.endswith(".nc")]
        if not nc_files:
            print(f"Scenario network file not found in {scenario_dir}, skipping.")
            return None
        
        network_path = os.path.join(scenario_dir, nc_files[0])
        
    else: # If a custom path is provided, where the network name is the same as the scenario name
        # Define the scenario directory
        scenario_dir = path

        # Get scenario network file
        network_path = f"{scenario_dir}/{scenario}.nc"
        if not os.path.exists(network_path):
            print(f"{scenario} network file not found in {scenario_dir}, skipping.")
            return None

    return network_path

def get_ci_costs(network, by_country=False, by_component=False, by_carrier=False):
    """Get costs and revenues from the network statistics, grouped by country, component, or carrier."""

    def get_ci(df):
        if "ci" in df.columns:
            return df.loc[df.ci != "", "ci"]
        return pd.Series(dtype=str)

    # Concatenate all ci values from generators, links, and storage_units
    all_ci = pd.concat([
        get_ci(network.generators),
        get_ci(network.links),
        get_ci(network.storage_units),
        get_ci(network.loads)
    ])

    # Get statistics as a DataFrame
    df = pd.DataFrame({
        "revenue": network.statistics.revenue(nice_names=False, groupby=["name", "carrier", "country"]),
        "capex": network.statistics.capex(nice_names=False, groupby=["name", "carrier", "country"]),
        "opex": network.statistics.opex(nice_names=False, groupby=["name", "carrier", "country"]),
    }).fillna(0)/1e6 # EUR to MEUR

    df["total_cost"] = df["capex"] + df["opex"]
    df["net_profit"] = df["revenue"] - df["total_cost"]
    cols = df.columns.tolist()

    # Filter rows where 'name' index level is in all_ci index, then map 'name' to corresponding ci
    df = df.loc[df.index.get_level_values("name").isin(all_ci.index)]

    # Create result with components grouped by metrics
    costs_CI = df.groupby(["component", "country"]).sum().unstack(level=0).swaplevel(0, 1, axis=1)
    components = costs_CI.columns.get_level_values(0).unique()
    ordered_cols = [(comp, metric) for comp in components for metric in cols if (comp, metric) in costs_CI.columns]

    if by_country:
        print("Total costs and revenues by country (MEUR):")
        country_df = costs_CI[ordered_cols]
        display(country_df.style.format("{:.2e}"))

    if by_component:
        print("Total system costs and revenues by component (bEUR):")
        component_df = costs_CI[ordered_cols].sum().unstack()/1e3
        display(component_df.style.format("{:.2e}"))

    if by_carrier:
        print("Total system costs and revenues by carrier (bEUR):")
        carrier_df = df.groupby(["carrier"]).sum()/1e3
        display(carrier_df.style.format("{:.2e}"))

In [9]:
# 1 - Retrieve results
planning_horizon_list = ["2025", "2030"]
time_resolution = "1"
ci_participation = "25"

for planning_horizon in planning_horizon_list:
    print(f"--------Planning Horizon: {planning_horizon}--------")

    scenarios = [f"vol-match-{planning_horizon}-ci{ci_participation}-{time_resolution}H",
                f"vol-match-{planning_horizon}-country-ci{ci_participation}-{time_resolution}H", # only for 2030
                f"247-cfe-{planning_horizon}-ci{ci_participation}-cfe100-{time_resolution}H",
                f"247-cfe-{planning_horizon}-ci{ci_participation}-cfe90-{time_resolution}H",
                f"emi-match-{planning_horizon}-ci{ci_participation}-data-aer-{time_resolution}H", # only for 2025
                f"emi-match-{planning_horizon}-ci{ci_participation}-data-mber-{time_resolution}H", # only for 2025
                f"emi-match-{planning_horizon}-ci{ci_participation}-data-moer-{time_resolution}H", # only for 2025
                f"emi-match-{planning_horizon}-ci{ci_participation}-data-cmer-{time_resolution}H", # only for 2025
                f"emi-match-{planning_horizon}-ci{ci_participation}-model-aer-{time_resolution}H", # only for 2030
                f"emi-match-{planning_horizon}-ci{ci_participation}-model-mber-{time_resolution}H", # only for 2030
                f"emi-match-{planning_horizon}-ci{ci_participation}-model-moer-{time_resolution}H", # only for 2030
                f"emi-match-{planning_horizon}-ci{ci_participation}-model-cmer-{time_resolution}H", # only for 2030
                f"vol-match-{planning_horizon}-NoResTargets-ci{ci_participation}-{time_resolution}H", # only for 2030
                ]
    

    for scenario in scenarios:
        network_path = get_network_path(scenario, path = False)

        if network_path is None: # if the scenario is not found (either the folder or the network in the folder)
            continue

        network = pypsa.Network(network_path)
        print(f"----Scenario: {scenario}----")
        get_ci_costs(network, by_country=False, by_component=True, by_carrier=False)

--------Planning Horizon: 2025--------


INFO:pypsa.io:Imported network vol-match-2025-ci25-1H.nc has buses, carriers, generators, global_constraints, lines, links, loads, storage_units, stores


----Scenario: vol-match-2025-ci25-1H----
Total system costs and revenues by component (bEUR):


,revenue,capex,opex,total_cost,net_profit
component,,,,,
Generator,1.93e+01,2.11e+01,1.09e-02,2.11e+01,-1.81e+00
Load,-1.98e+05,0.00e+00,0.00e+00,0.00e+00,-1.98e+05
StorageUnit,-2.43e-06,3.68e-06,3.90e-13,3.68e-06,-6.11e-06


Scenario directory ../results/vol-match-2025-country-ci25-1H/networks not found, skipping.


...

----Scenario: emi-match-2030-ci25-model-cmer-1H----
Total system costs and revenues by component (bEUR):


,revenue,capex,opex,total_cost,net_profit
component,,,,,
Generator,1.28e+01,2.03e+01,1.09e-02,2.03e+01,-7.48e+00
Load,-3.33e+06,0.00e+00,0.00e+00,0.00e+00,-3.33e+06
StorageUnit,-1.35e-06,1.80e-06,0.00e+00,1.80e-06,-3.15e-06


INFO:pypsa.io:Imported network vol-match-2030-NoResTargets-ci25-1H.nc has buses, carriers, generators, global_constraints, lines, links, loads, storage_units, stores


----Scenario: vol-match-2030-NoResTargets-ci25-1H----
Total system costs and revenues by component (bEUR):


,revenue,capex,opex,total_cost,net_profit
component,,,,,
Generator,1.61e+01,1.80e+01,1.05e-02,1.80e+01,-1.95e+00
Load,-3.68e+05,0.00e+00,0.00e+00,0.00e+00,-3.68e+05
StorageUnit,-2.45e-06,2.92e-06,3.40e-13,2.92e-06,-5.37e-06
